# 03 — Audit de la correction N4 (60 sujets d'entraînement)

Étape 3.1 de la [ROADMAP](../ROADMAP.md). La correction N4 (`preproc/bias.py`) a été appliquée à la FLAIR et à la T1 brutes des 60 sujets d'entraînement. Ce notebook **cherche les anomalies** et les explique.

Référence de comparaison : la correction SPM12 des organisateurs (`pre/`), dont on déduit le champ par `brute / corrigée`.

| Section | Contenu |
|---|---|
| 1 | Métriques d'audit et calcul sur les 60 sujets |
| 2 | Vue d'ensemble par scanner et sujets signalés |
| 3 | Anomalie 1 : à Singapour, le champ FLAIR de N4 ne ressemble pas à celui de SPM12 |
| 4 | Signal 2 : le champ T1 de Singapour est très étendu |
| 5 | Signal 3 : deux masques de tête « trop petits » (sub-107, sub-112) |
| 6 | Constat positif : les variations entre coupes d'Utrecht |
| 7 | Conclusions |

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from nibabel.processing import resample_from_to
from scipy import ndimage

from wmh_multisite.config import load_config, resolve_path
from wmh_multisite.utils import io

cfg = load_config()
B = resolve_path(cfg, "bids")
D = resolve_path(cfg, "derivatives") / "wmh-multisite"
TABLE = resolve_path(cfg, "results") / "tables" / "n4_audit.csv"
participants = pd.read_csv(B / "participants.tsv", sep="\t").set_index("participant_id")
report = pd.read_csv(D / "n4_report.csv")
COLORS = {"utrecht_philips_3t": "tab:blue", "singapore_siemens_3t": "tab:orange", "amsterdam_ge_3t": "tab:green"}

def paths(sub):
    a, ch, man = B / sub / "anat", B / "derivatives/challenge" / sub / "anat", B / "derivatives/manual" / sub / "anat"
    d = D / sub / "anat"
    return {"flair": a / f"{sub}_FLAIR.nii.gz", "t1": a / f"{sub}_T1w.nii.gz",
            "flair_spm": ch / f"{sub}_desc-spm12_FLAIR.nii.gz",
            "t1_flairgrid": ch / f"{sub}_space-FLAIR_T1w.nii.gz", "t1_spm": ch / f"{sub}_space-FLAIR_desc-spm12_T1w.nii.gz",
            "flair_n4": d / f"{sub}_desc-n4_FLAIR.nii.gz", "flair_field": d / f"{sub}_desc-biasfield_FLAIR.nii.gz",
            "t1_n4": d / f"{sub}_desc-n4_T1w.nii.gz", "t1_field": d / f"{sub}_desc-biasfield_T1w.nii.gz",
            "flair_head": d / f"{sub}_space-FLAIR_desc-head_mask.nii.gz", "t1_head": d / f"{sub}_space-T1w_desc-head_mask.nii.gz",
            "wmh": man / f"{sub}_space-FLAIR_desc-O1_dseg.nii.gz"}
print(report.status.value_counts().to_dict(), "| sujets :", report.participant_id.nunique())

In [ ]:
# Display helpers: canonical orientation (display only), physical aspect ratio, head-mask contour.
def canonical(path_or_img):
    img = io.load(path_or_img) if not isinstance(path_or_img, nib.Nifti1Image) else path_or_img
    can = nib.as_closest_canonical(img)
    return can.get_fdata(dtype=np.float32), can.header.get_zooms()[:3]

def take(vol, plane, frac):
    # slice of a canonical (RAS) volume: axial / coronal / sagittal at a relative position
    if plane == "axial":
        k = int(frac * (vol.shape[2] - 1)); return vol[:, :, k].T, (0, 1)
    if plane == "coronal":
        k = int(frac * (vol.shape[1] - 1)); return vol[:, k, :].T, (0, 2)
    k = int(frac * (vol.shape[0] - 1)); return vol[k, :, :].T, (1, 2)

def show_image_and_mask(ax, img, mask, plane, frac, title):
    v, z = canonical(img); m, _ = canonical(mask)
    sl, (a, b) = take(v, plane, frac); ms, _ = take(m, plane, frac)
    ax.imshow(sl, origin="lower", cmap="gray", vmin=0, vmax=np.percentile(v[v > 0], 99.5), aspect=z[b] / z[a])
    ax.contourf(ms, levels=[0.5, 1.5], colors=["gold"], alpha=0.25)
    ax.contour(ms, levels=[0.5], colors=["gold"], linewidths=0.8)
    ax.set_title(title, fontsize=8); ax.axis("off")

def show_field(ax, field, mask, plane, frac, title, vmin=0.75, vmax=1.25):
    f, z = canonical(field); m, _ = canonical(mask)
    sl, (a, b) = take(f, plane, frac); ms, _ = take(m, plane, frac)
    sl = np.ma.masked_where(ms < 0.5, sl / np.median(f[m > 0.5]))
    im = ax.imshow(sl, origin="lower", cmap="coolwarm", vmin=vmin, vmax=vmax, aspect=z[b] / z[a])
    ax.set_title(title, fontsize=8); ax.axis("off")
    return im

## 1. Métriques d'audit

Pour chaque sujet :

| Métrique | Définition | Attendu |
|---|---|---|
| `field_p5`, `field_p95` | étendue du champ N4 dans le masque de tête (rapportée à sa médiane) | ~0,8 à ~1,25 |
| `corr_flair_all` | corrélation entre log(champ N4) et log(champ SPM12) de la FLAIR, dans la tête | proche de 1 si les deux méthodes voient le même biais |
| `corr_flair_core` | idem, restreint au **cœur du cerveau** (masque érodé de 20 voxels dans le plan, moitié centrale des coupes) | idem, moins sensible aux bords |
| `corr_t1` | idem pour la T1 (notre champ, estimé à 1 mm, ramené sur la grille FLAIR, contre le champ SPM12 de la T1 alignée) | idem |
| `field_strength_n4`, `field_strength_spm` | coefficient de variation du champ dans la tête : **amplitude** du biais estimé | — |
| `cov_raw`, `cov_spm`, `cov_n4` | coefficient de variation de l'intensité du tissu non lésé dans le cœur du cerveau | doit diminuer après correction |
| `slice_spread_*` | dispersion, d'une coupe à l'autre, du 75e percentile du tissu non lésé (variation de luminosité entre coupes) | doit diminuer après correction |

Le calcul prend environ 5 minutes et est mis en cache dans `results/tables/n4_audit.csv` (statistiques uniquement, aucune image). Supprimer le fichier pour recalculer.

In [ ]:
def core_mask(head):
    core = np.stack([ndimage.binary_erosion(head[:, :, k], iterations=20) for k in range(head.shape[2])], 2)
    keep = np.zeros_like(core)
    keep[:, :, head.shape[2] // 4: 3 * head.shape[2] // 4] = True
    return core & keep

def audit_subject(sub):
    p = paths(sub)
    raw, spm, fld = io.load_data(p["flair"]), io.load_data(p["flair_spm"]), io.load_data(p["flair_field"])
    head = io.load_labels(p["flair_head"], allowed=(0, 1)).astype(bool)
    wmh = io.load_labels(p["wmh"])
    m = head & (raw > 0) & (spm > 0)
    core = core_mask(head) & m
    log_n4, log_spm = np.log(fld), np.log(np.where(m, raw / np.maximum(spm, 1e-6), 1))
    tissue = core & (wmh == 0)
    n4 = io.load_data(p["flair_n4"])
    zs = [k for k in range(raw.shape[2]) if tissue[:, :, k].sum() > 2000]
    def spread(x):
        v = np.array([np.percentile(x[:, :, k][tissue[:, :, k]], 75) for k in zs])
        return float(np.std(v / np.median(v)))
    spm_field = raw[m] / spm[m]
    # T1: our field (T1 grid) resampled onto the FLAIR grid vs SPM12 field of the aligned T1
    t1f = resample_from_to(io.load(p["t1_field"]), io.load(p["flair"]), order=1).get_fdata()
    o, s = io.load_data(p["t1_flairgrid"]), io.load_data(p["t1_spm"])
    mt = (o > np.percentile(o, 60)) & (s > 0) & (t1f > 0)
    t1head = io.load_labels(p["t1_head"], allowed=(0, 1)).astype(bool)
    t1fld = io.load_data(p["t1_field"])
    rel = lambda f, mk: np.percentile(f[mk] / np.median(f[mk]), [5, 95])
    fp, tp = rel(fld, head), rel(t1fld, t1head)
    ml = float(np.prod(io.load(p["flair"]).header.get_zooms()[:3])) / 1000
    return {"participant_id": sub, "scanner": participants.loc[sub, "scanner"],
            "flair_mask": float(head.mean()), "flair_head_ml": float(head.sum() * ml),
            "flair_field_p5": fp[0], "flair_field_p95": fp[1], "t1_field_p5": tp[0], "t1_field_p95": tp[1],
            "corr_flair_all": float(np.corrcoef(log_n4[m], log_spm[m])[0, 1]),
            "corr_flair_core": float(np.corrcoef(log_n4[core], log_spm[core])[0, 1]),
            "corr_t1": float(np.corrcoef(np.log(t1f[mt]), np.log(o[mt] / s[mt]))[0, 1]),
            "field_strength_n4": float(fld[m].std() / fld[m].mean()),
            "field_strength_spm": float(spm_field.std() / spm_field.mean()),
            "cov_raw": float(raw[tissue].std() / raw[tissue].mean()),
            "cov_spm": float(spm[tissue].std() / spm[tissue].mean()),
            "cov_n4": float(n4[tissue].std() / n4[tissue].mean()),
            "slice_spread_raw": spread(raw), "slice_spread_spm": spread(spm), "slice_spread_n4": spread(n4)}

if TABLE.is_file():
    audit = pd.read_csv(TABLE)
else:
    audit = pd.DataFrame([audit_subject(s) for s in sorted(report.participant_id.unique())])
    audit.round(4).to_csv(TABLE, index=False)
print(len(audit), "sujets audités")

## 2. Vue d'ensemble par scanner

In [ ]:
cols = ["flair_field_p5", "flair_field_p95", "t1_field_p5", "t1_field_p95", "corr_flair_all", "corr_flair_core", "corr_t1",
        "field_strength_n4", "field_strength_spm", "cov_raw", "cov_spm", "cov_n4",
        "slice_spread_raw", "slice_spread_spm", "slice_spread_n4", "flair_head_ml"]
audit.groupby("scanner")[cols].median().T.round(3)

In [ ]:
flags = {
    "corrélation FLAIR (tête) < 0,3": audit.corr_flair_all < 0.3,
    "corrélation FLAIR (cœur) < 0,5": audit.corr_flair_core < 0.5,
    "corrélation T1 < 0,5": audit.corr_t1 < 0.5,
    "N4 dégrade l'homogénéité (CoV > brut)": audit.cov_n4 > audit.cov_raw * 1.01,
    "champ hors [0,7 ; 1,4]": (audit[["flair_field_p5", "t1_field_p5"]].min(axis=1) < 0.7)
                               | (audit[["flair_field_p95", "t1_field_p95"]].max(axis=1) > 1.4),
    "masque FLAIR < 12 % du volume": audit.flair_mask < 0.12,
}
summary = pd.DataFrame({name: f.groupby(audit.scanner).sum() for name, f in flags.items()}).T
summary["sujets"] = [", ".join(audit[f].participant_id.str[4:]) for f in flags.values()]
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (col, title) in zip(axes, [("corr_flair_all", "corrélation N4 / SPM12 — FLAIR, tête"),
                                    ("corr_flair_core", "corrélation N4 / SPM12 — FLAIR, cœur"),
                                    ("corr_t1", "corrélation N4 / SPM12 — T1")]):
    for i, (sc, g) in enumerate(audit.groupby("scanner")):
        ax.scatter(np.full(len(g), i) + np.random.default_rng(0).uniform(-0.15, 0.15, len(g)), g[col],
                   color=COLORS[sc], s=18)
    ax.set_xticks(range(3), [s.split("_")[0] for s in sorted(audit.scanner.unique())])
    ax.axhline(0, color="k", lw=0.5); ax.set_ylim(-0.6, 1); ax.set_title(title, fontsize=10)
plt.tight_layout(); plt.show()

**Lecture.** Aucune image n'est dégradée par N4 (homogénéité toujours égale ou meilleure). La T1 s'accorde bien avec SPM12 partout. Le seul désaccord systématique est la **FLAIR de Singapour** : 19 sujets sur 20 ont une corrélation dans la tête inférieure à 0,3, contre environ 0,8 ailleurs.

## 3. Anomalie 1 : le champ FLAIR de Singapour

### 3a. Les deux champs, côte à côte

Champ N4 et champ SPM12 (`brute / corrigée`) pour un sujet de Singapour et un sujet d'Utrecht, **même échelle de couleur**. Les coupes affichées sont en orientation canonique (affichage seulement).

In [ ]:
def field_images(sub):
    p = paths(sub)
    raw, spm = io.load_data(p["flair"]), io.load_data(p["flair_spm"])
    head = io.load_labels(p["flair_head"], allowed=(0, 1)).astype(bool) & (raw > 0) & (spm > 0)
    n4 = io.load_data(p["flair_field"])
    spm_field = np.where(head, raw / np.maximum(spm, 1e-6), np.nan)
    n4_field = np.where(head, n4, np.nan)
    norm = lambda f: f / np.nanmedian(f)
    aff = io.load(p["flair"]).affine
    can = lambda v: nib.as_closest_canonical(nib.Nifti1Image(v.astype(np.float32), aff))
    return can(norm(n4_field)), can(norm(spm_field))

def show_fields(subs):
    fig, axes = plt.subplots(len(subs), 4, figsize=(15, 3.6 * len(subs)), squeeze=False)
    for r, sub in enumerate(subs):
        for c, (name, img) in enumerate(zip(["N4", "SPM12"], field_images(sub))):
            v = img.get_fdata(); z = img.header.get_zooms()[:3]
            for j, (sl, asp, plane) in enumerate([(v[:, :, v.shape[2] // 2], z[1] / z[0], "axiale"),
                                                  (v[:, v.shape[1] // 2, :], z[2] / z[0], "coronale")]):
                ax = axes[r, 2 * j + c]
                im = ax.imshow(np.ma.masked_invalid(sl.T), origin="lower", cmap="coolwarm", vmin=0.75, vmax=1.25, aspect=asp)
                ax.set_title(f"{sub} ({participants.loc[sub, 'scanner'].split('_')[0]}) — {name}, {plane}", fontsize=8)
                ax.axis("off")
    fig.colorbar(im, ax=axes, shrink=0.6, label="champ / médiane")
    plt.show()

show_fields(["sub-051", "sub-000"])

À Utrecht, les deux méthodes trouvent un motif voisin (centre plus clair, gradient vertical). À Singapour, le champ de **SPM12 est presque plat** alors que N4 estime un champ marqué, avec une zone centrale claire : le désaccord vient de là.

### 3b. Ma première hypothèse : le visage et le cou dans le masque — **réfutée**

La FLAIR de Singapour couvre beaucoup de visage et de cou (40 ml de tissu en médiane dans la zone défacée de la T1). Si ces tissus, très différents du cerveau, perturbaient N4, restreindre le masque devrait rétablir l'accord.

**D'abord, voir les masques.** Le masque de tête utilisé par N4 (en jaune) sur la FLAIR, pour trois sujets de Singapour et trois sujets des autres sites. Colonnes : coupe axiale basse (25 % de la hauteur du volume), coupe axiale médiane, coupe sagittale médiane.

In [ ]:
examples = ["sub-051", "sub-054", "sub-060", "sub-000", "sub-006", "sub-102"]
views = [("axial", 0.25, "axiale basse"), ("axial", 0.5, "axiale médiane"), ("sagittal", 0.5, "sagittale")]
fig, axes = plt.subplots(len(examples), 3, figsize=(11, 3.2 * len(examples)))
for r, sub in enumerate(examples):
    p = paths(sub)
    site = participants.loc[sub, "scanner"].split("_")[0]
    ml = audit.set_index("participant_id").loc[sub, "flair_head_ml"]
    for c, (plane, frac, name) in enumerate(views):
        show_image_and_mask(axes[r, c], p["flair"], p["flair_head"], plane, frac,
                            f"{sub} ({site}, masque {ml:.0f} ml) — {name}")
plt.tight_layout(); plt.show()

**Ce qu'on voit : le masque n'a pas la même définition selon le site.**
- **Utrecht** : le masque couvre **toute la tête**, cuir chevelu compris. Sur cette FLAIR, le seuil d'Otsu retient surtout le cuir chevelu et la graisse (les plus clairs) ; le remplissage des trous ajoute tout l'intérieur.
- **Amsterdam** : le masque couvre **essentiellement le cerveau**. Le cuir chevelu, sombre sur cette FLAIR 3D, reste sous le seuil.
- **Singapour** : entre les deux, et **irrégulier** : cerveau, morceaux de cuir chevelu, tissus du visage et des sinus dans les coupes basses, et des zones rectangulaires remplies (le remplissage des trous coupe par coupe ferme des cavités qui ne sont fermées que dans un plan).

Une même règle (seuil d'Otsu + remplissage) produit donc des masques différents selon le contraste de chaque scanner. Quelle part du masque est remplie par le remplissage des trous (voxels sous le seuil), et quelle part est quasiment vide (air) ?

In [ ]:
from skimage.filters import threshold_otsu
rows = []
for sub, sc in zip(audit.participant_id, audit.scanner):
    p = paths(sub)
    raw = io.load_data(p["flair"]); head = io.load_labels(p["flair_head"], allowed=(0, 1)).astype(bool)
    t = threshold_otsu(raw)
    rows.append({"scanner": sc, "part remplie (sous le seuil)": (head & (raw <= t)).sum() / head.sum(),
                 "part quasi vide (< 10 % du tissu)": (head & (raw < 0.1 * np.median(raw[raw > t]))).sum() / head.sum()})
fill = pd.DataFrame(rows)
fill.groupby("scanner").agg(["median", "max"]).round(3)

C'est **Utrecht** qui a le plus de voxels remplis (médiane ~30 %), alors que c'est là que N4 et SPM12 s'accordent le mieux ; et la part d'air réel dans le masque reste faible partout (médiane < 4 %). L'irrégularité des masques de Singapour **n'explique donc pas** à elle seule l'anomalie. Le test suivant le confirme : refaire N4 sur deux sujets de Singapour et un d'Utrecht avec deux masques restreints (environ 30 secondes).

In [ ]:
import ants
from wmh_multisite.preproc.bias import head_mask

rows = []
for sub in ["sub-051", "sub-054", "sub-000"]:
    p = paths(sub)
    img = ants.image_read(str(p["flair"]))
    raw, spm = img.numpy(), io.load_data(p["flair_spm"])
    head = head_mask(raw)
    m = head & (raw > 0) & (spm > 0)
    log_spm = np.log(raw[m] / spm[m])
    eroded = np.stack([ndimage.binary_erosion(head[:, :, k], iterations=10) for k in range(head.shape[2])], 2)
    zs = np.where(head.any(axis=(0, 1)))[0]
    upper = head.copy(); upper[:, :, : zs[0] + (zs[-1] - zs[0]) // 3] = False
    row = {"sujet": sub, "scanner": participants.loc[sub, "scanner"].split("_")[0],
           "masque de tête (actuel)": np.corrcoef(np.log(io.load_data(p["flair_field"])[m]), log_spm)[0, 1]}
    for name, mask in [("tête érodée (sans peau ni bords)", eroded), ("2/3 supérieurs (sans cou)", upper)]:
        f = ants.n4_bias_field_correction(img, mask=img.new_image_like(mask.astype(np.float32)), return_bias_field=True)
        row[name] = np.corrcoef(np.log(f.numpy()[m]), log_spm)[0, 1]
    rows.append(row)
pd.DataFrame(rows).round(3)

Changer le masque ne rétablit pas l'accord à Singapour (et dégrade Utrecht quand on retire le bas du volume). **Le masque n'est pas la cause.**

### 3c. Ce que montrent l'amplitude des champs et l'homogénéité du tissu

Deux mesures complémentaires :
- **l'amplitude du champ estimé** par chaque méthode (coefficient de variation du champ dans la tête) ;
- **la luminosité du tissu d'une coupe à l'autre** dans l'image brute, puis après chaque correction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for sc, g in audit.groupby("scanner"):
    axes[0].scatter(g.field_strength_spm, g.field_strength_n4, color=COLORS[sc], label=sc, s=20)
axes[0].plot([0, 0.25], [0, 0.25], "k--", lw=0.6)
axes[0].set_xlabel("amplitude du champ SPM12 (CoV)"); axes[0].set_ylabel("amplitude du champ N4 (CoV)"); axes[0].legend(fontsize=8)
med = audit.groupby("scanner")[["slice_spread_raw", "slice_spread_spm", "slice_spread_n4"]].median()
med.plot.bar(ax=axes[1], rot=0)
axes[1].set_xticks(range(3), [s.split("_")[0] for s in med.index]); axes[1].set_ylabel("dispersion entre coupes (médiane)")
axes[1].legend(["brute", "SPM12", "N4"]); axes[1].set_yscale("log")
plt.tight_layout(); plt.show()
med.round(3)

**Lecture.**
- **SPM12 voit moins de biais à Singapour** (amplitude médiane 0,065, contre 0,081 à Utrecht et 0,101 à Amsterdam), et la FLAIR brute y est un peu plus homogène d'une coupe à l'autre (0,028 contre 0,038 et 0,043). L'écart est net mais pas extrême : les distributions se chevauchent.
- **N4, lui, estime un champ d'amplitude à peu près constante (~0,12) sur les trois sites** (nuage horizontal). Quand le biais réel est fort (Amsterdam), les deux méthodes s'accordent ; quand il est faible (Singapour), la part du champ N4 qui n'est pas du biais (motifs anatomiques ou artefacts absorbés par le lissage B-spline) devient prépondérante, et la corrélation avec SPM12 s'effondre.
- **Aucune dégradation mesurable** : l'homogénéité du tissu (`cov`) et la dispersion entre coupes sont, sur chaque site, meilleures avec N4 qu'avec SPM12 et que l'image brute.

**Conclusion.** Sans vérité terrain sur le champ réel, on ne peut pas dire laquelle des deux méthodes a raison à Singapour. Deux faits sont établis : le biais y est plus faible qu'ailleurs, et N4 y estime un champ dont une part n'est probablement pas du biais, sans que cela dégrade l'homogénéité. Hypothèse sur l'origine du biais faible (non vérifiable, les en-têtes DICOM ne sont pas publiés) : une normalisation d'intensité appliquée par le scanner (sur Siemens, l'option « Prescan Normalize »).

**Vérifications prévues** :
- refaire N4 avec le masque du cerveau HD-BET (étape 3.3) et voir si l'accord avec SPM12 s'améliore ;
- l'expérience « effet du prétraitement » (étape 5.3) dira si cette différence change la segmentation, ce qui est la question qui compte.

## 4. Signal 2 : le champ T1 de Singapour est très étendu

Le champ N4 de la T1 de Singapour va jusqu'à 1,4 à 1,6 fois la médiane (contre ~1,25 ailleurs). Est-ce un problème de N4 ?

In [ ]:
print(audit.groupby("scanner")[["t1_field_p5", "t1_field_p95", "corr_t1"]].median().round(3))
sub = audit[audit.scanner == "singapore_siemens_3t"].sort_values("t1_field_p95").participant_id.iloc[-1]
p = paths(sub)
t1, n4, fld, head = (nib.as_closest_canonical(io.load(p[k])) for k in ("t1", "t1_n4", "t1_field", "t1_head"))
inside = head.get_fdata()[head.get_fdata().shape[0] // 2] > 0
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, (img, title, kw) in zip(axes, [(t1, "T1 brute", {"cmap": "gray"}), (n4, "T1 après N4", {"cmap": "gray"}),
                                       (fld, "champ N4", {"cmap": "coolwarm", "vmin": 0.6, "vmax": 1.6})]):
    v = img.get_fdata(); sl = v[v.shape[0] // 2]                     # sagittal slice
    if "vmin" not in kw:
        kw = {**kw, "vmin": 0, "vmax": np.percentile(sl, 99.5)}
    else:   # field shown inside the head mask only (outside, it is extrapolated)
        sl = np.ma.masked_where(~inside, sl / np.median(sl[inside]))
    ax.imshow(sl.T, origin="lower", **kw); ax.set_title(f"{sub} — {title} (sagittal)", fontsize=9); ax.axis("off")
plt.show()

**Comparaison directe avec SPM12.** Le champ SPM12 de la T1 n'existe que sur la grille FLAIR (les organisateurs ont corrigé la T1 *après* l'avoir ramenée sur cette grille) : on ramène donc notre champ N4, estimé à 1 mm, sur la même grille (affines seules, interpolation linéaire) et on affiche les deux côte à côte, même échelle de couleur, dans la zone de tissu de la T1. Trois sujets : le cas le plus étendu de Singapour, puis un sujet d'Utrecht et un d'Amsterdam.

In [ ]:
def t1_fields_on_flair_grid(sub):
    p = paths(sub)
    o, s_ = io.load_data(p["t1_flairgrid"]), io.load_data(p["t1_spm"])
    tissue = (o > np.percentile(o, 60)) & (s_ > 0)
    spm_field = np.where(tissue, o / np.maximum(s_, 1e-6), 0).astype(np.float32)
    n4_field = resample_from_to(io.load(p["t1_field"]), io.load(p["flair"]), order=1).get_fdata(dtype=np.float32)
    aff = io.load(p["flair"]).affine
    mk = lambda v: nib.Nifti1Image(v.astype(np.float32), aff)
    return mk(n4_field), mk(spm_field), mk(tissue.astype(np.float32))

subs = [audit[audit.scanner == "singapore_siemens_3t"].sort_values("t1_field_p95").participant_id.iloc[-1], "sub-000", "sub-100"]
views = [("axial", 0.5, "axiale"), ("coronal", 0.5, "coronale"), ("sagittal", 0.5, "sagittale")]
fig, axes = plt.subplots(len(subs), 6, figsize=(19, 3.4 * len(subs)))
for r, sub in enumerate(subs):
    n4f, spmf, tissue = t1_fields_on_flair_grid(sub)
    corr = audit.set_index("participant_id").loc[sub, "corr_t1"]
    site = participants.loc[sub, "scanner"].split("_")[0]
    for c, (plane, frac, name) in enumerate(views):
        im = show_field(axes[r, 2 * c], n4f, tissue, plane, frac, f"{sub} ({site}) — N4, {name}", 0.6, 1.4)
        show_field(axes[r, 2 * c + 1], spmf, tissue, plane, frac, f"SPM12, {name} (corr. {corr:.2f})", 0.6, 1.4)
fig.colorbar(im, ax=axes, shrink=0.5, label="champ / médiane")
plt.show()

Sur la T1, les deux méthodes trouvent le **même motif** sur les trois sujets (mêmes zones claires et sombres) : à Singapour, un côté de la tête bien plus clair ; à Utrecht, le centre ; à Amsterdam, l'arrière, plus marqué pour N4 que pour SPM12 (d'où une corrélation un peu plus faible, 0,74). Le champ est fort mais **réel** : SPM12 trouve le même motif (corrélation médiane 0,89, la meilleure des trois sites). Les valeurs extrêmes sont près de la surface de la tête, là où la sensibilité des antennes de réception est la plus forte : un profil typique de biais de réception. **Pas une anomalie de N4.**

## 5. Signal 3 : deux masques de tête « trop petits »

sub-107 et sub-112 ont un masque FLAIR couvrant moins de 12 % du volume. En volume absolu :

In [ ]:
a = audit[audit.scanner == "amsterdam_ge_3t"].sort_values("flair_head_ml")
print(a[["participant_id", "flair_mask", "flair_head_ml"]].head(4).round(3).to_string(index=False))
print("médiane Amsterdam GE3T : %.0f ml" % a.flair_head_ml.median())
plt.figure(figsize=(8, 3))
for sc, g in audit.groupby("scanner"):
    plt.hist(g.flair_head_ml, bins=15, alpha=0.6, color=COLORS[sc], label=sc)
plt.xlabel("volume du masque de tête FLAIR (ml)"); plt.legend(fontsize=8); plt.show()

**Les voir.** Les deux cas signalés, et sub-100 (même scanner, masque de taille habituelle) comme référence. Pour chacun : la FLAIR et son masque de tête (jaune), puis le champ N4 dans ce masque.

In [ ]:
cases = ["sub-107", "sub-112", "sub-100"]
fig, axes = plt.subplots(len(cases), 4, figsize=(15, 3.6 * len(cases)))
for r, sub in enumerate(cases):
    p = paths(sub)
    row = audit.set_index("participant_id").loc[sub]
    t = f"{sub} ({row.flair_mask:.1%} du volume, {row.flair_head_ml:.0f} ml)"
    show_image_and_mask(axes[r, 0], p["flair"], p["flair_head"], "axial", 0.5, t + " — axiale")
    show_image_and_mask(axes[r, 1], p["flair"], p["flair_head"], "sagittal", 0.5, "sagittale")
    im = show_field(axes[r, 2], p["flair_field"], p["flair_head"], "axial", 0.5, "champ N4, axiale")
    show_field(axes[r, 3], p["flair_field"], p["flair_head"], "sagittal", 0.5, "champ N4, sagittale")
fig.colorbar(im, ax=axes[:, 2:], shrink=0.5, label="champ / médiane")
plt.show()

Les masques de sub-107 et sub-112 sont **corrects** : ils suivent le cerveau, comme partout à Amsterdam (section 3). Leur volume (~1 160 ml) est proche d'un volume intracrânien ; celui de sub-100 est plus grand surtout parce qu'il inclut, en plus, une partie du cuir chevelu à l'arrière et des tissus du visage. Leurs champs N4 sont réguliers. **Fausse alerte**, avec deux leçons :
- une fraction du volume dépend du champ de vue : raisonner en ml ;
- même en ml, le volume du masque de tête **n'est pas comparable d'un site à l'autre**, puisque sa définition change (tête entière à Utrecht, cerveau à Amsterdam). Pour toute mesure de volume, il faudra le masque du cerveau HD-BET (3.3).

## 6. Constat positif : les variations de luminosité entre coupes

Sur les trois sites, N4 réduit davantage que SPM12 la variation de luminosité du tissu d'une coupe à l'autre (médianes de la section 3c : Amsterdam 0,043 brute, 0,034 SPM12, 0,024 N4 ; Utrecht 0,038, 0,024, 0,017 ; Singapour 0,028, 0,022, 0,017). Exemple sur le sujet d'Utrecht au plus fort gradient vertical :

In [ ]:
sub = audit[audit.scanner == "utrecht_philips_3t"].sort_values("slice_spread_raw").participant_id.iloc[-1]
p = paths(sub)
raw, spm, n4 = (io.load_data(p[k]) for k in ("flair", "flair_spm", "flair_n4"))
tissue = core_mask(io.load_labels(p["flair_head"], allowed=(0, 1)).astype(bool)) & (io.load_labels(p["wmh"]) == 0)
zs = [k for k in range(raw.shape[2]) if tissue[:, :, k].sum() > 2000]
plt.figure(figsize=(9, 3.5))
for v, name in [(raw, "brute"), (spm, "SPM12"), (n4, "N4")]:
    prof = np.array([np.percentile(v[:, :, k][tissue[:, :, k]], 75) for k in zs])
    plt.plot(zs, prof / np.median(prof), "o-", ms=3, label=name)
plt.xlabel("indice de coupe (bas -> haut)"); plt.ylabel("luminosité du tissu / médiane"); plt.legend()
plt.title(f"{sub} — luminosité du tissu non lésé, coupe par coupe"); plt.show()

## 7. Conclusions

| Constat | Nature | Action |
|---|---|---|
| Champ FLAIR de Singapour différent de SPM12 (19/20 sujets) | Biais réel plus faible à Singapour ; N4 y estime un champ dont une part n'est probablement pas du biais. Aucune dégradation de l'homogénéité | **Question ouverte**, tracée dans J-030 : revérifier avec le masque HD-BET (3.3) et par l'expérience « effet du prétraitement » (5.3) |
| Champ T1 de Singapour étendu (p95 jusqu'à 1,6) | Biais réel, confirmé par SPM12 (corrélation 0,89) | Aucune |
| Masques FLAIR < 12 % (sub-107, sub-112) | Fausse alerte : masques corrects (cerveau), grand champ de vue | Au QC, raisonner en ml, pas en fraction du volume |
| Définition du masque de tête variable selon le site (tête entière, cerveau, intermédiaire irrégulier) | Effet du contraste propre à chaque scanner sur le seuil d'Otsu ; sans conséquence mesurable sur N4 | Ne jamais utiliser ce masque pour des volumes ou des statistiques ; masque du cerveau HD-BET pour cela (3.3, J-028) |
| Variations de luminosité entre coupes | N4 les réduit davantage que SPM12 sur les trois sites | — |
| Rapport `n4_report.csv` sans statistiques pour les sujets déjà traités | Défaut de `bias.py` | Corrigé : les statistiques sont recalculées depuis les fichiers |

**Aucune image n'est dégradée par N4** ; la correction peut alimenter le recalage (3.2) et l'export nnU-Net (3.4).